# Questão 15 - Aplicação com ontologia em OWL 2

## 1. Aplicação

A aplicação é uma ontologia de um departamento universitário. Ela guarda professores, alunos, disciplinas e projetos de pesquisa, e é usada para responder perguntas que dependem de raciocínio, por exemplo: quem pode orientar alunos de doutorado, quais alunos são colegas de laboratório e se a oferta de disciplinas do semestre tem algum erro.

A ferramenta usada foi o Owlready2, em Python, com o reasoner HermiT. O arquivo gerado (`universidade.owl`) pode ser aberto no Protégé, onde o mesmo HermiT está disponível no menu Reasoner.

O foco é mostrar recursos que entraram na versão 2 do OWL: cadeias de propriedades, restrições de cardinalidade qualificada, restrições sobre tipos de dados, propriedades irreflexivas e assimétricas e classes disjuntas em grupo.

In [1]:
import pandas as pd
from owlready2 import (
    AllDifferent, AllDisjoint, AsymmetricProperty, ConstrainedDatatype, DataProperty, FunctionalProperty,
    IrreflexiveProperty, ObjectProperty, OwlReadyInconsistentOntologyError, PropertyChain,
    SymmetricProperty, Thing, TransitiveProperty, World, sync_reasoner,
)

mundo = World()
onto = mundo.get_ontology("http://exemplo.org/universidade.owl")

## 2. Representação do conhecimento

**Classes**

Pessoa se divide em Professor e Aluno. Aluno se divide em AlunoGraduacao, AlunoMestrado e AlunoDoutorado. Há ainda Disciplina, ProjetoPesquisa e Laboratorio.

Professor e Aluno não são declarados disjuntos de propósito: um aluno de doutorado pode dar aula como professor substituto. Já os três níveis de aluno são disjuntos entre si.

**Propriedades**

ministra e ministradaPor: professor e disciplina (inversas).

cursa: aluno e disciplina.

orienta e orientadoPor: professor e aluno (inversas). orienta é irreflexiva e assimétrica: ninguém orienta a si mesmo e, se A orienta B, B não orienta A.

preRequisitoDe: entre disciplinas. É transitiva.

membroDe: pessoa e laboratório.

coordena: professor e projeto.

participaDe: pessoa e projeto.

colegaDe: simétrica, entre pessoas.

professorDe: liga o professor aos alunos das suas disciplinas. É deduzida pela cadeia ministra o inverso de cursa.

titulacao, anoIngresso e cargaHoraria: propriedades de dados.

In [2]:
with onto:
    class Pessoa(Thing): pass
    class Professor(Pessoa): pass
    class Aluno(Pessoa): pass
    class AlunoGraduacao(Aluno): pass
    class AlunoMestrado(Aluno): pass
    class AlunoDoutorado(Aluno): pass
    class Disciplina(Thing): pass
    class ProjetoPesquisa(Thing): pass
    class Laboratorio(Thing): pass

    AllDisjoint([Pessoa, Disciplina, ProjetoPesquisa, Laboratorio])
    AllDisjoint([AlunoGraduacao, AlunoMestrado, AlunoDoutorado])

    class ministra(ObjectProperty):
        domain = [Professor]
        range = [Disciplina]

    class ministradaPor(ObjectProperty):
        inverse_property = ministra

    class cursa(ObjectProperty):
        domain = [Aluno]
        range = [Disciplina]

    class cursadaPor(ObjectProperty):
        inverse_property = cursa

    class orienta(ObjectProperty, IrreflexiveProperty, AsymmetricProperty):
        domain = [Professor]
        range = [Aluno]

    class orientadoPor(ObjectProperty):
        inverse_property = orienta

    class preRequisitoDe(ObjectProperty, TransitiveProperty):
        domain = [Disciplina]
        range = [Disciplina]

    class membroDe(ObjectProperty):
        domain = [Pessoa]
        range = [Laboratorio]

    class coordena(ObjectProperty):
        domain = [Professor]
        range = [ProjetoPesquisa]

    class participaDe(ObjectProperty):
        domain = [Pessoa]
        range = [ProjetoPesquisa]

    class colegaDe(ObjectProperty, SymmetricProperty):
        domain = [Pessoa]
        range = [Pessoa]

    class professorDe(ObjectProperty):
        domain = [Professor]
        range = [Aluno]

    class titulacao(DataProperty, FunctionalProperty):
        domain = [Professor]
        range = [str]

    class anoIngresso(DataProperty, FunctionalProperty):
        domain = [Aluno]
        range = [int]

    class cargaHoraria(DataProperty, FunctionalProperty):
        domain = [Disciplina]
        range = [int]

    professorDe.property_chain.append(PropertyChain([ministra, cursadaPor]))
    participaDe.property_chain.append(PropertyChain([orientadoPor, coordena]))

## 3. Axiomas e classes definidas

As regras do departamento são escritas como axiomas.

**Toda disciplina é ministrada por exatamente um professor.** É uma cardinalidade qualificada, recurso do OWL 2: a restrição conta só os valores de uma classe específica.

**Quem é orientado pelo coordenador de um projeto participa do projeto.** É a cadeia orientadoPor o coordena, declarada na célula anterior.

**Professor doutor:** professor com titulação igual a "doutorado".

**Orientador de doutorado:** professor que orienta algum aluno de doutorado. Um axioma diz que todo orientador de doutorado precisa ser professor doutor. Se alguém sem doutorado orientar um doutorando, a ontologia fica inconsistente.

**Professor sobrecarregado:** professor que ministra três ou mais disciplinas (cardinalidade mínima qualificada).

**Disciplina extensa:** disciplina com carga horária de 90 horas ou mais (restrição sobre tipo de dado).

**Aluno veterano:** aluno que ingressou até 2022.

**Aluno pesquisador:** aluno que participa de algum projeto de pesquisa.

In [3]:
with onto:
    Disciplina.is_a.append(ministradaPor.exactly(1, Professor))

    class ProfessorDoutor(Professor):
        equivalent_to = [Professor & titulacao.value("doutorado")]

    class OrientadorDoutorado(Professor):
        equivalent_to = [Professor & orienta.some(AlunoDoutorado)]

    OrientadorDoutorado.is_a.append(ProfessorDoutor)

    class ProfessorSobrecarregado(Professor):
        equivalent_to = [Professor & ministra.min(3, Disciplina)]

    class DisciplinaExtensa(Disciplina):
        equivalent_to = [Disciplina & cargaHoraria.some(ConstrainedDatatype(int, min_inclusive=90))]

    class AlunoVeterano(Aluno):
        equivalent_to = [Aluno & anoIngresso.some(ConstrainedDatatype(int, max_inclusive=2022))]

    class AlunoPesquisador(Aluno):
        equivalent_to = [Aluno & participaDe.some(ProjetoPesquisa)]

## 4. Indivíduos

Os dados do departamento são fictícios. Cada indivíduo é declarado só com a classe básica e com os fatos diretos.

Para o reasoner conseguir contar disciplinas, é preciso dizer que os indivíduos são diferentes entre si (AllDifferent). O OWL não assume que nomes diferentes indicam coisas diferentes.

In [4]:
with onto:
    lia = Laboratorio("LabIA")
    lsd = Laboratorio("LabSistemas")

    ia, ml, pln, calc, alg, redes = [
        Disciplina(n, cargaHoraria=h)
        for n, h in [("IA", 60), ("AprendizagemDeMaquina", 60), ("PLN", 60), ("Calculo", 90), ("Algoritmos", 90), ("Redes", 60)]
    ]
    alg.preRequisitoDe = [ia]
    ia.preRequisitoDe = [ml]
    ml.preRequisitoDe = [pln]

    helena = Professor("Helena", titulacao="doutorado", membroDe=[lia], ministra=[ia, ml, pln])
    marcos = Professor("Marcos", titulacao="doutorado", membroDe=[lsd], ministra=[redes])
    rita = Professor("Rita", titulacao="mestrado", membroDe=[lsd], ministra=[calc, alg])

    proj_kg = ProjetoPesquisa("GrafosDeConhecimento")
    proj_iot = ProjetoPesquisa("RedesDeSensores")
    helena.coordena = [proj_kg]
    marcos.coordena = [proj_iot]

    joao = AlunoDoutorado("Joao", anoIngresso=2021, membroDe=[lia], cursa=[pln])
    bia = AlunoMestrado("Bia", anoIngresso=2024, membroDe=[lia], cursa=[ml, pln])
    caio = AlunoGraduacao("Caio", anoIngresso=2022, cursa=[ia, redes])
    duda = AlunoGraduacao("Duda", anoIngresso=2025, cursa=[calc, alg])
    enzo = AlunoMestrado("Enzo", anoIngresso=2023, membroDe=[lsd], cursa=[redes])

    helena.orienta = [joao, bia]
    marcos.orienta = [enzo]
    joao.colegaDe = [bia]

    AllDifferent([ia, ml, pln, calc, alg, redes])
    AllDifferent([helena, marcos, rita, joao, bia, caio, duda, enzo])
    AllDifferent([proj_kg, proj_iot])
    AllDifferent([lia, lsd])

onto.save(file="universidade.owl", format="rdfxml")
print(len(list(onto.classes())), "classes,", len(list(onto.properties())), "propriedades,", len(list(onto.individuals())), "indivíduos")

15 classes, 15 propriedades, 18 indivíduos


## 5. Raciocínio

A função abaixo mostra as classes e algumas propriedades de cada pessoa. Ela é chamada antes e depois do reasoner.

In [5]:
def nomes(itens):
    return ", ".join(sorted(i.name for i in itens)) or "-"


def retrato(individuos, propriedades):
    return pd.DataFrame(
        {
            i.name: {"classes": nomes(c for c in i.is_a if hasattr(c, "name")), **{p: nomes(getattr(i, p)) for p in propriedades}}
            for i in individuos
        }
    ).T


pd.set_option("display.max_colwidth", None)
pessoas = [helena, marcos, rita, joao, bia, caio, duda, enzo]
retrato(pessoas, ["participaDe", "professorDe", "colegaDe"])

,classes,participaDe,professorDe,colegaDe
Helena,Professor,-,-,-
Marcos,Professor,-,-,-
Rita,Professor,-,-,-
Joao,AlunoDoutorado,-,-,Bia
Bia,AlunoMestrado,-,-,Joao
Caio,AlunoGraduacao,-,-,-
Duda,AlunoGraduacao,-,-,-
Enzo,AlunoMestrado,-,-,-


In [6]:
with onto:
    sync_reasoner(mundo, infer_property_values=True, debug=0)

print("Consistente. Classes insatisfatíveis:", list(mundo.inconsistent_classes()))
retrato(pessoas, ["participaDe", "professorDe", "colegaDe"])

Consistente. Classes insatisfatíveis: []


,classes,participaDe,professorDe,colegaDe
Helena,"OrientadorDoutorado, ProfessorSobrecarregado",-,"Bia, Caio, Joao",-
Marcos,ProfessorDoutor,-,"Caio, Enzo",-
Rita,Professor,-,Duda,-
Joao,"AlunoDoutorado, AlunoPesquisador, AlunoVeterano",GrafosDeConhecimento,-,Bia
Bia,"AlunoMestrado, AlunoPesquisador",GrafosDeConhecimento,-,Joao
Caio,"AlunoGraduacao, AlunoVeterano",-,-,-
Duda,AlunoGraduacao,-,-,-
Enzo,"AlunoMestrado, AlunoPesquisador",RedesDeSensores,-,-


**O que o reasoner concluiu**

Helena foi classificada como ProfessorDoutor (pela titulação), OrientadorDoutorado (orienta João, que é aluno de doutorado) e ProfessorSobrecarregado (ministra três disciplinas diferentes).

A classe ProfessorDoutor não aparece na linha de Helena porque a tabela mostra só as classes mais específicas, e OrientadorDoutorado já é subclasse de ProfessorDoutor. Marcos é ProfessorDoutor. Rita não recebeu nenhuma classe nova, pois tem mestrado e ministra duas disciplinas.

João e Caio são AlunoVeterano, pelo ano de ingresso.

João, Bia e Enzo são AlunoPesquisador. Ninguém declarou que eles participam de projetos. O reasoner chegou a isso em dois passos: o orientador coordena um projeto, a cadeia de propriedades coloca o orientando no projeto, e quem participa de projeto é pesquisador.

A propriedade professorDe foi toda preenchida pela cadeia ministra e cursadaPor.

Bia passou a ser colega de João pela simetria de colegaDe.

In [7]:
retrato([ia, ml, pln, calc, alg, redes], ["preRequisitoDe", "ministradaPor", "cursadaPor"])

,classes,preRequisitoDe,ministradaPor,cursadaPor
IA,Disciplina,"AprendizagemDeMaquina, PLN",Helena,Caio
AprendizagemDeMaquina,Disciplina,PLN,Helena,Bia
PLN,Disciplina,-,Helena,"Bia, Joao"
Calculo,DisciplinaExtensa,-,Rita,Duda
Algoritmos,DisciplinaExtensa,"AprendizagemDeMaquina, IA, PLN",Rita,Duda
Redes,Disciplina,-,Marcos,"Caio, Enzo"


Nas disciplinas, preRequisitoDe foi completada pela transitividade: Algoritmos é pré-requisito de IA, de Aprendizagem de Máquina e de PLN. Cálculo e Algoritmos viraram DisciplinaExtensa por terem 90 horas. As propriedades inversas ministradaPor e cursadaPor também foram preenchidas.

## 6. Consultas

Com as inferências prontas, as perguntas do início podem ser respondidas com consultas SPARQL. O trecho `rdfs:subClassOf*` faz a consulta aceitar também as subclasses. Helena, por exemplo, aparece como OrientadorDoutorado, que é subclasse de ProfessorDoutor.

**Quem pode orientar alunos de doutorado?**

In [8]:
PREFIXO = "PREFIX u: <http://exemplo.org/universidade.owl#>\n"


def sparql(consulta):
    return [[getattr(v, "name", v) for v in linha] for linha in mundo.sparql(PREFIXO + consulta)]


sparql("SELECT DISTINCT ?p WHERE { ?p a ?c . ?c rdfs:subClassOf* u:ProfessorDoutor . } ORDER BY ?p")

[['Helena'], ['Marcos']]

**Quais alunos participam de qual projeto, e quem coordena?**

In [9]:
sparql("""
SELECT DISTINCT ?aluno ?projeto ?coordenador WHERE {
    ?aluno a ?c ; u:participaDe ?projeto .
    ?c rdfs:subClassOf* u:AlunoPesquisador .
    ?coordenador u:coordena ?projeto .
} ORDER BY ?projeto ?aluno
""")

[['Joao', 'GrafosDeConhecimento', 'Helena'],
 ['Bia', 'GrafosDeConhecimento', 'Helena'],
 ['Enzo', 'RedesDeSensores', 'Marcos']]

**O que um aluno precisa ter cursado antes de PLN?**

In [10]:
sparql("SELECT ?d WHERE { ?d u:preRequisitoDe u:PLN . }")

[['IA'], ['AprendizagemDeMaquina'], ['Algoritmos']]

**Quais alunos têm aula com um professor sobrecarregado?**

In [11]:
sparql("SELECT DISTINCT ?aluno ?prof WHERE { ?prof a ?c ; u:professorDe ?aluno . ?c rdfs:subClassOf* u:ProfessorSobrecarregado . } ORDER BY ?aluno")

[['Joao', 'Helena'], ['Bia', 'Helena'], ['Caio', 'Helena']]

## 7. Detecção de erros

O reasoner também serve para validar os dados. Cada teste abaixo carrega a ontologia salva em um mundo separado, acrescenta um fato que quebra uma regra e executa o HermiT.

In [12]:
def testar(descricao, alterar):
    w = World()
    o = w.get_ontology("universidade.owl").load()
    with o:
        alterar(o)
    try:
        sync_reasoner(w, debug=0)
        print(f"{descricao}: consistente")
    except OwlReadyInconsistentOntologyError:
        print(f"{descricao}: INCONSISTENTE")


def rita_orienta_doutorando(o):
    o.Rita.orienta.append(o.Joao)


def duas_professoras(o):
    o.Rita.ministra.append(o.Redes)


def aluno_em_dois_niveis(o):
    o.Bia.is_a.append(o.AlunoDoutorado)


def orientacao_mutua(o):
    o.Joao.is_a.append(o.Professor)
    o.Joao.orienta.append(o.Helena)


testar("Ontologia original", lambda o: None)
testar("Rita (mestre) orienta um aluno de doutorado", rita_orienta_doutorando)
testar("Redes ministrada por Marcos e por Rita", duas_professoras)
testar("Bia declarada como mestrado e doutorado", aluno_em_dois_niveis)
testar("João orienta a própria orientadora", orientacao_mutua)

Ontologia original: consistente


Rita (mestre) orienta um aluno de doutorado: INCONSISTENTE


Redes ministrada por Marcos e por Rita: INCONSISTENTE


Bia declarada como mestrado e doutorado: INCONSISTENTE


João orienta a própria orientadora: INCONSISTENTE


Os quatro erros foram detectados, cada um por um axioma diferente.

No primeiro, Rita seria classificada como OrientadorDoutorado, que exige ser ProfessorDoutor. Como a titulação dela é funcional e vale "mestrado", não pode valer "doutorado".

No segundo, a disciplina teria dois professores diferentes, contra a cardinalidade exata de um.

No terceiro, as classes de nível são disjuntas.

No quarto, orienta é assimétrica.

## 8. Hipótese do mundo aberto

Um ponto importante do raciocínio em OWL aparece quando se tenta definir "professor que não orienta ninguém". Rita não orienta ninguém na base, mas o reasoner não a coloca nessa classe.

In [13]:
with onto:
    class ProfessorSemOrientandos(Professor):
        equivalent_to = [Professor & orienta.max(0, Aluno)]

    sync_reasoner(mundo, infer_property_values=True, debug=0)

print("ProfessorSemOrientandos:", nomes(onto.ProfessorSemOrientandos.instances()))

ProfessorSemOrientandos: -


A classe fica vazia. Para o OWL, não estar escrito que Rita orienta alguém não quer dizer que ela não orienta: pode ser só uma informação que falta. Esse comportamento é o contrário do de um banco de dados, que trata o que não está registrado como falso.

Para obter a conclusão é preciso fechar o mundo de forma explícita, declarando que Rita orienta no máximo zero alunos.

In [14]:
with onto:
    rita.is_a.append(orienta.max(0, Aluno))
    sync_reasoner(mundo, infer_property_values=True, debug=0)

print("ProfessorSemOrientandos:", nomes(onto.ProfessorSemOrientandos.instances()))

ProfessorSemOrientandos: Rita


## 9. Discussão

**Representação do conhecimento**

A ontologia separa três níveis. As classes e propriedades formam o vocabulário do domínio. Os axiomas guardam as regras do departamento. Os indivíduos guardam os fatos. As regras ficam declaradas na própria ontologia e não escondidas em código, então qualquer ferramenta compatível com OWL 2 chega às mesmas conclusões.

Os recursos do OWL 2 foram necessários em vários pontos. Sem cadeia de propriedades não haveria como dizer que o orientando participa do projeto do orientador. Sem cardinalidade qualificada não seria possível exigir exatamente um professor por disciplina sem restringir outros valores. Sem restrição de tipo de dado, as classes AlunoVeterano e DisciplinaExtensa teriam que ser preenchidas manualmente.

**Raciocínio**

O HermiT fez três tipos de trabalho: classificou indivíduos em classes definidas, deduziu relações que não foram declaradas e verificou a consistência. As consultas da seção 6 só dão a resposta certa depois dessas inferências.

**Limites**

O OWL 2 não faz contas. Não dá para definir, por exemplo, "professor com mais de 180 horas de aula somadas", porque isso exige somar valores.

A hipótese do mundo aberto exige cuidado e surpreende quem está acostumado com banco de dados.

O raciocínio fica caro com muitas restrições de cardinalidade e muitos indivíduos. Para bases grandes costuma-se usar perfis mais simples da linguagem, como OWL 2 EL ou OWL 2 RL.